In [12]:
# Basic
import pandas as pd
import numpy as np

# Progress bar
from tqdm import tqdm

# BERT embeddings
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize

# Optional: speed
import torch

print("Torch device:", "GPU" if torch.cuda.is_available() else "CPU")




Torch device: CPU


In [2]:
news_clean = pd.read_csv("data/cleaned/news_clean.csv")
news_clean.head()

,news_id,category,subcategory,title,abstract
0,N55528,lifestyle,lifestyleroyals,"The Brands Queen Elizabeth, Prince Charles, an...","Shop the notebooks, jackets, and more that the..."
1,N19639,health,weightloss,50 Worst Habits For Belly Fat,These seemingly harmless habits are holding yo...
2,N61837,news,newsworld,The Cost of Trump's Aid Freeze in the Trenches...,Lt. Ivan Molchanets peeked over a parapet of s...
3,N53526,health,voices,I Was An NBA Wife. Here's How It Affected My M...,"I felt like I was a fraud, and being an NBA wi..."
4,N38324,health,medical,"How to Get Rid of Skin Tags, According to a De...","They seem harmless, but there's a very good re..."


In [3]:
print(news_clean.shape)
print(news_clean.isnull().sum())

(51282, 5)
news_id           0
category          0
subcategory       0
title             0
abstract       2666
dtype: int64


In [4]:
news_clean["abstract"] = news_clean["abstract"].fillna("")

In [5]:
# Main content
news_clean["text"] = news_clean["title"] + " " + news_clean["abstract"]

# Category content
news_clean["cat_text"] = news_clean["category"] + " " + news_clean["subcategory"]

In [6]:
model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
def encode_in_batches(texts, model, batch_size=64):
    embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        batch_emb = model.encode(batch, show_progress_bar=False)
        embeddings.extend(batch_emb)
    
    return np.array(embeddings)

In [8]:
text_embeddings = encode_in_batches(
    news_clean["text"].tolist(), model
)

print("Text embedding shape:", text_embeddings.shape)

100%|██████████| 802/802 [01:44<00:00,  7.68it/s]

Text embedding shape: (51282, 384)


In [9]:
cat_embeddings = encode_in_batches(
    news_clean["cat_text"].tolist(), model
)

print("Category embedding shape:", cat_embeddings.shape)

100%|██████████| 802/802 [00:22<00:00, 35.81it/s]

Category embedding shape: (51282, 384)


In [10]:
final_embeddings = np.hstack([text_embeddings, cat_embeddings])

print("Final embedding shape:", final_embeddings.shape)

Final embedding shape: (51282, 768)


In [13]:
final_embeddings = normalize(final_embeddings)

In [14]:
# Save embeddings
np.save("data/embedded/news_embeddings_final.npy", final_embeddings)

# Save news IDs
news_ids = news_clean["news_id"].tolist()
np.save("data/embedded/news_ids.npy", news_ids)

# Save index mapping
news_clean["embedding_index"] = range(len(news_clean))
news_clean.to_csv("data/embedded/news_with_index.csv", index=False)

print("Saved all files successfully!")

Saved all files successfully!


In [15]:
print("Sample news_id:", news_ids[0])
print("Embedding (first 10 values):", final_embeddings[0][:10])
print("Embedding dimension:", final_embeddings.shape[1])

Sample news_id: N55528
Embedding (first 10 values): [-0.01886897  0.06061954  0.03854043  0.00348515  0.04707707  0.02564699
  0.01827295 -0.04234226 -0.01606141  0.01058363]
Embedding dimension: 768
